# Initial OH\*+H\* configurations on bare Ni nanoparticles and slabs

One builder for both uses of initial structures:

| `MODE` | Use |
|---|---|
| `"enumerate"` | Generate a dataset: every allowed O-site/H-site pair on the full surface, or in the local environment of one Ni atom |
| `"ranking"` | Build selected candidates of a `*_ranked_candidates.csv` (from `prediction.py`) for DFT validation |

Both modes share one site catalog, identical to `training_scripts/prediction.py`, so generated datasets and ranked
candidates describe the same kind of sites:

- **Surface atoms:** probe-sphere exposure ($r_\mathrm{Ni}$ = 1.24 Å, $r_\mathrm{probe}$ = 1.20 Å, 200 points); for slabs only the
  upper side ($z \geq z_\mathrm{mid}$).
- **Ni–Ni bonds:** distance ≤ 3.0 Å (minimum image for slabs).
- **Sites:** T = one surface atom; B = two bonded atoms; H = three mutually bonded atoms (closing locally, not through
  the cell) plus every 3-atom subset of an F site; F = locally closed 4-ring of bonds with no surface atom bonded to all four.
- **Site pairs (enumerate mode):** O and H\* never occupy the identical site. Everything else is optional screening:
  - `OVERLAP_RULES = True` applies the overlap rules of `prediction.py`: a T site may not sit on an atom of the other
    B/H/F site, a B site may not be an edge of the other H/F site, an H site may not be a face of the other F site;
  - `SITE_DIST_WINDOW = (min, max)` keeps only pairs whose O-site/H-site centroid distance lies in that range
    (`None` = no distance screening). The ranked CSVs used (1.8, 5.0) Å for NPs and lower minima for slabs.

  Whatever the screening, the distance checks below still skip structures where atoms actually clash.

**Placement.** O sits above the site centroid along the site normal at a fixed height per site type, with the OH hydrogen
0.98 Å further along the same normal. H\* sits above its site centroid at the height that gives a target mean Ni–H distance,
clamped to `H_HEIGHT_RANGE`. Site normals: +z for slabs; for NPs the local outward normal (opposite the mean Ni–Ni bond
direction of the anchors), made perpendicular to the bond for B sites and to the anchor plane for H/F sites.

**Checks on every built structure.** Minimum O–H\*, Ni–O and Ni–H\* distances, and a round trip: the Ni atoms within 2.45 Å
of O and within 2.10 Å of H\* must be the intended anchors. Structures failing the distance checks are skipped and logged.

**Output.** One extxyz per structure (`Lattice`, `pbc`, `config_label`, `motif_label`, `o_anchor`, `h_anchor`, as in the
`dataset_*_ini_stru.extxyz` files; ranking mode adds the CSV rank, score and energy), optional VASP files, and for slabs a
QE `ATOMIC_POSITIONS` block in crystal coordinates with the bottom layers frozen. Anchor indices are zero-based, as in the CSVs.

## 1. Settings

In [ ]:
from pathlib import Path

# ---------------------------------------------------------------- input / mode
BARE_STRUCTURE = Path("../ML_relaxed_bare_geom/MACE/testing/Ni130_MACE_opt.xyz")
SYSTEM_LABEL = "Ni130"          # used in config_label and file names
MODE = "ranking"                    # "ranking" | "enumerate"
OUTPUT_DIR = Path("built_configurations") / f"{SYSTEM_LABEL}_{MODE}"

# ---------------------------------------------------------------- ranking mode
RANKING_CSV = Path("../test_systems/Ni130/Ni130_ranked_candidates.csv")
SELECT_MODE = "first_N"             # "first_N" | "by_rank" | "from_file"
N_BUILD = 10                        # first_N: number of structures to build
FIRST_N_RETAINED_ONLY = True        # first_N: only candidates with s_ret >= RETENTION_MIN
RETENTION_MIN = 0.80
SELECT_RANKS = [1, 2, 3]            # by_rank: CSV ranks to build, in this order
CANDIDATES_FILE = Path("selected.csv")   # from_file: CSV with the same columns, every row is built

# ---------------------------------------------------------------- enumerate mode
# Where to build configurations:
#   None -> the full surface (every site of the catalog)
#   int  -> the local environment of that Ni atom (zero-based index in BARE_STRUCTURE):
#           every atom of every T/B/H/F site containing it (on (100)-like regions this includes
#           the diagonal atoms that close the fourfold sites); only sites made of these atoms are built
PATCH_CENTER = None
ENUMERATE_MOTIFS = None             # None = all motifs, or e.g. ["BH", "HH", "TH"]

# ---------------------------------------------------------------- site catalog (= prediction.py)
NI_NEIGHBOR_CUTOFF = 3.0
SURFACE_R_ATOM = 1.24
SURFACE_R_PROBE = 1.20
SURFACE_N_POINTS = 200
TOP_SURFACE_ONLY = True             # slabs: upper side only
TOP_AXIS = 2
RING_CLOSURE_TOL = 0.5
INCLUDE_HOLLOW4 = True
INCLUDE_SQUARE_DERIVED_HOLLOW = True   # the Ni100_4x4 ranking was run with False (no H motifs)

# ---------------------------------------------------------------- pair screening (enumerate mode)
# O and H* never share the identical site; the rest is optional.
OVERLAP_RULES = True                # prediction.py overlap rules (T on an atom of the other site, B edge of
                                    # the other H/F, H face of the other F)
SITE_DIST_WINDOW = None             # None, or (min, max) O-site/H-site centroid distance in Å, e.g. (1.8, 5.0);
                                    # the ranked CSVs used (1.8, 5.0) for NPs, lower minima for slabs

# ---------------------------------------------------------------- placement
OH_BOND = 0.98
O_HEIGHT = {"T": 1.90, "B": 1.40, "H": 1.65, "F": 1.40}         # O height above the site centroid (Å)
H_TARGET_NI_DISTANCE = {"T": 1.60, "B": 1.80, "H": 1.90, "F": 1.80}  # target mean Ni-H* distance (Å)
H_HEIGHT_RANGE = (0.55, 1.40)                                    # clamp on the B/H/F H* height (Å);
                                                                 # top-site H* sits exactly at its target
SKIP_IF_H_TARGET_IMPOSSIBLE = True  # target shorter than the site's lateral radius
NP_BOX = 25.0                       # cubic cell written around NPs (Å); the NP is centred in it

# ---------------------------------------------------------------- checks
MIN_DIST_O_HSTAR = 1.35
MIN_DIST_NI_O = 1.60
MIN_DIST_NI_HSTAR = 1.30
ROUNDTRIP_NI_O_CUTOFF = 2.45
ROUNDTRIP_NI_H_CUTOFF = 2.10

# ---------------------------------------------------------------- output
WRITE_VASP = False
WRITE_QE_POSITIONS = True           # slabs only
N_FROZEN_LAYERS = 2                 # slabs: bottom Ni layers frozen in the QE block
LAYER_GAP = 0.5                     # Å; Ni atoms closer than this in z belong to one layer

## 2. Structure and geometry helpers

In [ ]:
import ast
import json
from itertools import combinations

import numpy as np
import pandas as pd
from ase import Atom
from ase.geometry import find_mic
from ase.io import read, write

atoms0 = read(BARE_STRUCTURE, index=-1)
if set(atoms0.get_chemical_symbols()) != {"Ni"}:
    raise ValueError(f"{BARE_STRUCTURE} must contain only Ni atoms")
PERIODIC = bool(np.any(atoms0.pbc)) and atoms0.cell.volume > 1e-6
if not PERIODIC:
    atoms0.pbc = False
POS = atoms0.get_positions()
N_NI = len(atoms0)
CENTER = POS.mean(axis=0)
print(f"{BARE_STRUCTURE.name}: {N_NI} Ni, {'slab (periodic)' if PERIODIC else 'nanoparticle (non-periodic)'}")


def mic(vectors) -> np.ndarray:
    """Minimum-image Cartesian vectors (unchanged for nanoparticles)."""
    vectors = np.asarray(vectors, dtype=float)
    if not PERIODIC:
        return vectors
    single = vectors.ndim == 1
    out = find_mic(np.atleast_2d(vectors), atoms0.cell, atoms0.pbc)[0]
    return out[0] if single else out


def mic_norm(vectors) -> np.ndarray:
    return np.linalg.norm(mic(vectors), axis=-1)


def unwrap(site, ref=None) -> np.ndarray:
    """Anchor coordinates, each the image closest to ref (default: first anchor)."""
    ref = POS[site[0]] if ref is None else np.asarray(ref, dtype=float)
    return ref + mic(POS[list(site)] - ref)


def centroid(site, ref=None) -> np.ndarray:
    return unwrap(site, ref).mean(axis=0)


def half_cell_ambiguous(site) -> bool:
    """True if an anchor is half a lattice vector from the first one (image undefined)."""
    if not PERIODIC:
        return False
    frac = np.linalg.solve(atoms0.cell.array.T, (POS[list(site)] - POS[site[0]]).T).T
    frac -= np.round(frac)
    return bool(np.any(np.abs(np.abs(frac[:, atoms0.pbc]) - 0.5) < 0.02))

## 3. Surface atoms, bonds and site catalog

In [ ]:
def fibonacci_sphere(n: int) -> np.ndarray:
    k = np.arange(n, dtype=float)
    z = 1.0 - 2.0 * (k + 0.5) / n
    r = np.sqrt(np.clip(1.0 - z * z, 0.0, None))
    phi = np.pi * (3.0 - np.sqrt(5.0)) * k
    return np.column_stack((r * np.cos(phi), r * np.sin(phi), z))


def surface_atoms_by_exposure() -> list[int]:
    """Atoms with any probe-accessible point on their r_atom + r_probe shell."""
    sphere = fibonacci_sphere(SURFACE_N_POINTS)
    shell = SURFACE_R_ATOM + SURFACE_R_PROBE
    surface = []
    for i in range(N_NI):
        near = np.flatnonzero(mic_norm(POS - POS[i]) <= 2.0 * shell)
        near = near[near != i]
        points = POS[i] + shell * sphere
        buried = np.zeros(SURFACE_N_POINTS, dtype=bool)
        for j in near:
            buried |= mic_norm(points - POS[j]) < shell
            if buried.all():
                break
        if (~buried).any():
            surface.append(i)
    if PERIODIC and TOP_SURFACE_ONLY:
        z = POS[:, TOP_AXIS]
        z_mid = 0.5 * (z.min() + z.max())
        surface = [i for i in surface if POS[i, TOP_AXIS] >= z_mid]
    return surface


# All Ni-Ni bonds (also used for the local normals) and the surface subgraph.
BONDED = [set(np.flatnonzero((mic_norm(POS - POS[i]) <= NI_NEIGHBOR_CUTOFF)
                             & (np.arange(N_NI) != i)).tolist()) for i in range(N_NI)]
SURFACE = surface_atoms_by_exposure()
SURFACE_SET = set(SURFACE)
NEIGHBORS = {i: BONDED[i] & SURFACE_SET for i in SURFACE}


def ring_closes_locally(cycle) -> bool:
    """False for a periodic row that only closes by wrapping around the cell."""
    total = sum(mic(POS[b] - POS[a]) for a, b in zip(cycle, cycle[1:] + cycle[:1]))
    return float(np.linalg.norm(total)) < RING_CLOSURE_TOL


def build_site_catalog() -> dict[str, list[tuple[int, ...]]]:
    """Same sites as prediction.generate_site_catalog, found by walking the bond graph
    (testing every 4-atom combination is intractable for large NPs)."""
    tops = [(i,) for i in SURFACE]
    bridges = sorted({tuple(sorted((i, j))) for i in SURFACE for j in NEIGHBORS[i]})
    triangles = set()
    for a in SURFACE:
        for b in NEIGHBORS[a]:
            for c in NEIGHBORS[a] & NEIGHBORS[b]:
                tri = tuple(sorted((a, b, c)))
                if tri not in triangles and ring_closes_locally(list(tri)):
                    triangles.add(tri)
    squares = set()
    if INCLUDE_HOLLOW4:
        # 4-ring a-b-d-c-a with no diagonal bonds (exactly four edges, degree two).
        for a in SURFACE:
            for b, c in combinations(sorted(NEIGHBORS[a]), 2):
                if c in NEIGHBORS[b]:
                    continue
                for d in (NEIGHBORS[b] & NEIGHBORS[c]) - {a}:
                    if d in NEIGHBORS[a]:
                        continue
                    quad = tuple(sorted((a, b, c, d)))
                    if quad in squares or not ring_closes_locally([a, b, d, c]):
                        continue
                    if set.intersection(*(NEIGHBORS[x] for x in quad)) - set(quad):
                        continue  # capped/folded 4-ring
                    squares.add(quad)
    squares = sorted(squares)
    hollows = set(triangles)
    if INCLUDE_SQUARE_DERIVED_HOLLOW:
        hollows |= {tuple(sorted(t)) for sq in squares for t in combinations(sq, 3)}
    return {"T": tops, "B": bridges, "H": sorted(hollows), "F": squares}


CATALOG = build_site_catalog()
SITE_SET = {kind: set(sites) for kind, sites in CATALOG.items()}
print(f"Surface atoms: {len(SURFACE)} of {N_NI}")
print("Sites: " + ", ".join(f"{k} {len(v)}" for k, v in CATALOG.items()))

In [ ]:
SITE_TYPE = {1: "T", 2: "B", 3: "H", 4: "F"}


def site_separation(o_site, h_site) -> float:
    return float(mic_norm(centroid(h_site) - centroid(o_site)))


def pair_allowed(o_site, h_site) -> bool:
    """O and H* never on the identical site; optional overlap rules and distance window."""
    o_set, h_set = set(o_site), set(h_site)
    if o_set == h_set:
        return False
    if OVERLAP_RULES:
        o_type, h_type = SITE_TYPE[len(o_site)], SITE_TYPE[len(h_site)]
        if o_type == "T" and h_type in "BHF" and o_set <= h_set:
            return False
        if h_type == "T" and o_type in "BHF" and h_set <= o_set:
            return False
        if o_type == "B" and h_type in "HF" and o_set <= h_set:
            return False
        if h_type == "B" and o_type in "HF" and h_set <= o_set:
            return False
        if o_type == "H" and h_type == "F" and o_set <= h_set:
            return False
        if h_type == "H" and o_type == "F" and h_set <= o_set:
            return False
    if SITE_DIST_WINDOW is not None:
        low, high = SITE_DIST_WINDOW
        if not low <= site_separation(o_site, h_site) <= high:
            return False
    return True


## 4. Placement and checks

In [ ]:
def outward_normal(i: int) -> np.ndarray:
    """Opposite the mean bond direction of atom i (points out of the particle)."""
    mean_bond = mic(POS[list(BONDED[i])] - POS[i]).mean(axis=0)
    return -mean_bond / np.linalg.norm(mean_bond)


def site_frame(site, ref=None) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Unwrapped anchor positions, centroid and outward unit normal of a site."""
    points = unwrap(site, ref)
    center = points.mean(axis=0)
    if PERIODIC:
        up = np.zeros(3)
        up[TOP_AXIS] = 1.0
    else:
        up = sum(outward_normal(a) for a in site)
        up /= np.linalg.norm(up)
    if len(site) == 1 or (PERIODIC and len(site) == 2):
        normal = up
    elif len(site) == 2:
        bond = (points[1] - points[0]) / np.linalg.norm(points[1] - points[0])
        normal = up - np.dot(up, bond) * bond
        normal /= np.linalg.norm(normal)
    else:
        normal = np.linalg.svd(points - center)[2][-1]   # normal of the anchor plane
        if np.dot(normal, up) < 0:
            normal = -normal
    return points, center, normal


def h_height(site_type: str, points, center) -> tuple[float, str | None]:
    """H* height for the target mean Ni-H* distance; second item is a note if clamped."""
    target = H_TARGET_NI_DISTANCE[site_type]
    if site_type == "T":
        return target, None
    lateral2 = float(np.mean(np.sum((points - center) ** 2, axis=1)))
    if lateral2 >= target**2:
        if SKIP_IF_H_TARGET_IMPOSSIBLE:
            raise ValueError(f"H* target {target:.2f} A below the site's lateral radius "
                             f"{np.sqrt(lateral2):.2f} A")
        return H_HEIGHT_RANGE[0], "target impossible: minimum height used"
    height = float(np.sqrt(target**2 - lateral2))
    clamped = min(max(height, H_HEIGHT_RANGE[0]), H_HEIGHT_RANGE[1])
    note = None
    if abs(clamped - height) > 1e-9:
        note = (f"H* height clamped {height:.2f} -> {clamped:.2f} A "
                f"(mean Ni-H* {np.sqrt(lateral2 + clamped**2):.2f} A instead of {target:.2f} A)")
    return clamped, note


def place(o_site, h_site) -> tuple:
    """Bare structure + O, H(OH), H* for one site pair; returns (atoms, notes)."""
    for site in (o_site, h_site):
        if half_cell_ambiguous(site):
            raise ValueError(f"site {site} spans half a cell: periodic image undefined")
    o_type, h_type = SITE_TYPE[len(o_site)], SITE_TYPE[len(h_site)]
    _, o_center, o_normal = site_frame(o_site)
    o_pos = o_center + O_HEIGHT[o_type] * o_normal
    h_points, h_center, h_normal = site_frame(h_site)
    height, note = h_height(h_type, h_points, h_center)
    atoms = atoms0.copy()
    atoms.append(Atom("O", o_pos))
    atoms.append(Atom("H", o_pos + OH_BOND * o_normal))
    atoms.append(Atom("H", h_center + height * h_normal))
    return atoms, [note] if note else []


def check_structure(atoms, o_site, h_site) -> tuple[list[str], list[str]]:
    """Errors (structure skipped) and warnings (structure kept) for a built structure."""
    pos = atoms.get_positions()
    o, h_oh, h_star = N_NI, N_NI + 1, N_NI + 2
    d_ni_o = mic_norm(POS - pos[o])
    d_ni_h = mic_norm(POS - pos[h_star])
    d_o_hstar = float(mic_norm(pos[h_star] - pos[o]))
    errors, warnings = [], []
    if d_o_hstar < MIN_DIST_O_HSTAR:
        errors.append(f"O-H* {d_o_hstar:.2f} A < {MIN_DIST_O_HSTAR}")
    if float(mic_norm(pos[h_star] - pos[h_oh])) < MIN_DIST_O_HSTAR:
        errors.append("H(OH)-H* too close")
    if d_ni_o.min() < MIN_DIST_NI_O:
        errors.append(f"Ni-O {d_ni_o.min():.2f} A < {MIN_DIST_NI_O}")
    if d_ni_h.min() < MIN_DIST_NI_HSTAR:
        errors.append(f"Ni-H* {d_ni_h.min():.2f} A < {MIN_DIST_NI_HSTAR}")
    o_found = tuple(np.flatnonzero(d_ni_o < ROUNDTRIP_NI_O_CUTOFF).tolist())
    h_found = tuple(np.flatnonzero(d_ni_h < ROUNDTRIP_NI_H_CUTOFF).tolist())
    if o_found != tuple(sorted(o_site)):
        warnings.append(f"Ni within {ROUNDTRIP_NI_O_CUTOFF} A of O: {o_found} (intended {tuple(sorted(o_site))})")
    if h_found != tuple(sorted(h_site)):
        warnings.append(f"Ni within {ROUNDTRIP_NI_H_CUTOFF} A of H*: {h_found} (intended {tuple(sorted(h_site))})")
    return errors, warnings

## 5. Writers

In [ ]:
def finalize_cell(atoms):
    """Slabs keep their cell and PBC; NPs are centred in a cubic NP_BOX cell, pbc = F."""
    if PERIODIC:
        atoms.pbc = True
        return atoms
    atoms.set_cell([NP_BOX] * 3)
    atoms.translate(0.5 * NP_BOX - CENTER)
    atoms.pbc = False
    return atoms


def frozen_bottom_layers(atoms) -> list[int]:
    z = atoms.positions[:N_NI, TOP_AXIS]
    order = np.argsort(z)
    layer_of = np.zeros(N_NI, dtype=int)
    for previous, current in zip(order, order[1:]):
        layer_of[current] = layer_of[previous] + (z[current] - z[previous] > LAYER_GAP)
    return [int(i) for i in np.flatnonzero(layer_of < N_FROZEN_LAYERS)]


def site_label(atoms) -> str:
    """Motif and zero-based anchors, e.g. 'motif=BH o_anchor=[115, 123] h_anchor=[16, 97, 123]'."""
    info = atoms.info
    return (f"{info['config_label']} motif={info['motif_label']} "
            f"o_anchor={np.atleast_1d(info['o_anchor']).tolist()} "
            f"h_anchor={np.atleast_1d(info['h_anchor']).tolist()} (zero-based)")


def write_structure(atoms, name: str) -> None:
    write(OUTPUT_DIR / f"{name}.extxyz", atoms, format="extxyz")
    if WRITE_VASP:
        vasp = OUTPUT_DIR / f"{name}.vasp"
        write(vasp, atoms, format="vasp", direct=True, sort=False)
        lines = vasp.read_text().splitlines()
        lines[0] = site_label(atoms)  # POSCAR comment line
        vasp.write_text("\n".join(lines) + "\n")
    if PERIODIC and WRITE_QE_POSITIONS:
        frozen = set(frozen_bottom_layers(atoms))
        qe_dir = OUTPUT_DIR / "qe_atomic_positions"
        qe_dir.mkdir(exist_ok=True)
        lines = [f"! {site_label(atoms)}", "ATOMIC_POSITIONS crystal"]
        for i, (symbol, frac) in enumerate(zip(atoms.get_chemical_symbols(),
                                               atoms.get_scaled_positions(wrap=True))):
            flags = "  0 0 0" if i in frozen else ""
            lines.append(f"  {symbol:2s} {frac[0]:16.10f} {frac[1]:16.10f} {frac[2]:16.10f}{flags}")
        (qe_dir / f"{name}.txt").write_text("\n".join(lines) + "\n")


def build_one(o_site, h_site, name: str, info: dict) -> dict:
    """Place, check, annotate and write one structure; returns a log record."""
    record = {"name": name, "motif": SITE_TYPE[len(o_site)] + SITE_TYPE[len(h_site)],
              "o_anchor": list(o_site), "h_anchor": list(h_site), **info}
    try:
        atoms, notes = place(o_site, h_site)
        errors, warnings = check_structure(atoms, o_site, h_site)
    except ValueError as exc:
        return record | {"status": "skipped", "reason": str(exc)}
    if errors:
        return record | {"status": "skipped", "reason": "; ".join(errors)}
    atoms.info = {"config_label": name, "motif_label": record["motif"],
                  "o_anchor": np.array(sorted(o_site)), "h_anchor": np.array(sorted(h_site)),
                  "o_h_site_distance_A": round(site_separation(o_site, h_site), 4),
                  **{k: v for k, v in info.items() if v is not None}}
    write_structure(finalize_cell(atoms), name)
    return record | {"status": "built", "reason": "; ".join(notes + warnings)}

## 6a. Ranking mode

The CSV anchors are used as they are, never remapped. Each requested site must exist in the catalog of section 3,
the same catalog `prediction.py` scores. A row whose site is missing (for example from a different bare geometry
or different surface settings) is skipped and logged.

In [ ]:
def parse_anchors(value) -> tuple[int, ...]:
    return tuple(sorted(int(v) for v in ast.literal_eval(str(value))))


def select_rows() -> pd.DataFrame:
    if SELECT_MODE == "from_file":
        return pd.read_csv(CANDIDATES_FILE)
    ranking = pd.read_csv(RANKING_CSV).sort_values("rank")
    if SELECT_MODE == "by_rank":
        rows = ranking.set_index("rank").reindex(SELECT_RANKS)
        missing = rows.index[rows["motif"].isna()].tolist()
        if missing:
            print(f"Ranks not in {RANKING_CSV.name}: {missing}")
        return rows.dropna(subset=["motif"]).reset_index()
    if SELECT_MODE == "first_N":
        if FIRST_N_RETAINED_ONLY:
            ranking = ranking[ranking["retention_probability"] >= RETENTION_MIN]
        return ranking
    raise ValueError(f"Unknown SELECT_MODE {SELECT_MODE!r}")


def run_ranking_mode() -> pd.DataFrame:
    log = []
    for _, row in select_rows().iterrows():
        if SELECT_MODE == "first_N" and sum(r["status"] == "built" for r in log) >= N_BUILD:
            break
        o_site, h_site = parse_anchors(row["o_anchor_idx"]), parse_anchors(row["h_anchor_idx"])
        name = f"{SYSTEM_LABEL}_rank{int(row['rank']):05d}_{row['motif']}"
        info = {"source_rank": int(row["rank"]), "candidate_id": int(row.get("candidate_id", -1)),
                "pred_Eads_eV": float(row["pred_Eads_eV"]),
                "retention_probability": float(row["retention_probability"])}
        problems = [f"{label} site {site} not in the catalog"
                    for label, site in (("O", o_site), ("H*", h_site))
                    if site not in SITE_SET[SITE_TYPE[len(site)]]]
        if SITE_TYPE[len(o_site)] + SITE_TYPE[len(h_site)] != row["motif"]:
            problems.append(f"motif {row['motif']} does not match the anchor counts")
        if not problems and "site_distance_A" in row and \
                abs(site_separation(o_site, h_site) - float(row["site_distance_A"])) > 0.05:
            problems.append("O-H* site distance differs from the CSV value")
        if problems:
            log.append({"name": name, "motif": row["motif"], "o_anchor": list(o_site),
                        "h_anchor": list(h_site), **info, "status": "skipped",
                        "reason": "; ".join(problems)})
            continue
        log.append(build_one(o_site, h_site, name, info))
    return pd.DataFrame(log)

## 6b. Enumerate mode

All allowed site pairs of the catalog, either on the full surface (`PATCH_CENTER = None`) or in the local
environment of one Ni atom (`PATCH_CENTER = index`). The local environment is every atom of every T/B/H/F site that
contains the centre atom: its bonded surface neighbours, plus, where the centre is a corner of a fourfold site, the
diagonal atom that closes it. Only sites made entirely of these atoms are built.

In [ ]:
def patch_atoms() -> set[int] | None:
    """None for the full surface, else every atom of every site containing PATCH_CENTER."""
    if PATCH_CENTER is None:
        return None
    if PATCH_CENTER not in SURFACE_SET:
        raise ValueError(f"PATCH_CENTER Ni {PATCH_CENTER} is not a surface atom; "
                         f"surface atoms: {sorted(SURFACE)}")
    patch = {a for kind in "TBHF" for site in CATALOG[kind] if PATCH_CENTER in site for a in site}
    print(f"Local environment of Ni {PATCH_CENTER}: {sorted(patch)}")
    return patch


def enumerate_pairs(patch: set[int] | None) -> list[tuple[tuple[int, ...], tuple[int, ...]]]:
    sites = [s for kind in "TBHF" for s in CATALOG[kind] if patch is None or set(s) <= patch]
    pairs = [(o, h) for o in sites for h in sites if pair_allowed(o, h)]
    if ENUMERATE_MOTIFS is not None:
        pairs = [(o, h) for o, h in pairs if SITE_TYPE[len(o)] + SITE_TYPE[len(h)] in ENUMERATE_MOTIFS]
    return pairs


def run_enumerate_mode() -> pd.DataFrame:
    pairs = enumerate_pairs(patch_atoms())
    print(f"{len(pairs)} allowed site pairs")
    return pd.DataFrame([build_one(o, h, f"{SYSTEM_LABEL}_conf{k:05d}", {})
                         for k, (o, h) in enumerate(pairs)])

## 7. Run

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
log = run_ranking_mode() if MODE == "ranking" else run_enumerate_mode()
log.to_csv(OUTPUT_DIR / "build_log.csv", index=False)
(OUTPUT_DIR / "settings.json").write_text(json.dumps(
    {k: (str(v) if isinstance(v, Path) else v) for k, v in globals().items()
     if k.isupper() and isinstance(v, (str, int, float, bool, list, tuple, dict, type(None), Path))
     and k not in {"POS", "CATALOG", "SITE_SET", "SURFACE", "NEIGHBORS", "BONDED", "SITE_TYPE"}},
    indent=1, default=str))

built = log[log["status"] == "built"]
print(f"Built {len(built)} structures in {OUTPUT_DIR}; skipped {int((log['status'] == 'skipped').sum())}")
print("Built per motif:", built["motif"].value_counts().to_dict())
if (log["status"] == "skipped").any():
    print("\nSkipped:")
    print(log.loc[log["status"] == "skipped", ["name", "reason"]].to_string(index=False))
notes = built[built["reason"].astype(bool)]
if len(notes):
    print(f"\n{len(notes)} built structures with notes (clamped H* height or anchors not recovered):")
    print(notes["reason"].value_counts().head(10).to_string())